*   Generalization is the ability of a trained model to perform well on new examples, not just the examples it saw during training.

In [1]:
import math
import random
import numpy as np
import torch

# 3.6.1 Training Error and Generalization Error

## 1. Intuition

*   Training error is error measured on the data used for learning.
*   Generalization error is expected error on new data from the same real problem.
*   Because we cannot see all future data, we estimate it with validation or test data.

## 2. Why this exists

*   A model that memorizes training data may have low training error but poor real-world performance.
*   Generalization is the reason evaluation data matters.

## 3. Examples

*   Compute training and validation loss separately.

In [2]:
train_pred = torch.tensor([1.0, 2.0, 3.0])
train_y = torch.tensor([1.0, 2.0, 4.0])

valid_pred = torch.tensor([1.0, 2.0])
valid_y = torch.tensor([1.0, 2.0])

train_loss = ((train_pred - train_y) ** 2).mean() # (1-1) ^^ 2 + (2-2) ^^ 2 + (4-3) ^^ 2 = 1; 1/3 = 1/3
valid_loss = ((valid_pred - valid_y) ** 2).mean() # (1-1) ^^ 2 + (2-2)= 0; 0/2 = 0

train_loss, valid_loss # 1/3, 0

(tensor(0.3333), tensor(0.))

## 4. Step-by-step breakdown

*   The training loss uses predictions and labels from training examples.

*   The validation loss uses examples not used for parameter updates.

*   Comparing these values helps diagnose whether the model is only fitting the training data.

## 5. Connection to ML systems

*   ML systems usually split data into training and validation or test sets.
*   The split gives a less biased estimate of future performance.



## 6. Common confusion points

- Low training error alone is not enough.
- Validation data should not be used for gradient updates.
- Test data should be touched as little as possible.
- Generalization error is estimated, not known exactly (since you cannot introduce all generalizable data to the model at once).
> *   We cannot measure true real-world performance directly because future data is unknown.
> *   Instead, we use validation/test sets as samples to estimate how well the model will generalize.



# 3.6.2 Underfitting or Overfitting?

## 1. Intuition

*   Underfitting means the model is too simple or poorly trained to fit even the training data.
> * Generally, underfitting means poor performance on both training and validation data.
> * Validation loss outperforming training loss is likely due to randomness, data differences, or regularization.
*   Overfitting means the model fits training data too specifically and performs worse on new data.



## 2. Why this exists

*   These diagnoses guide what to change. Underfitting and overfitting require different fixes.

## 3. Examples

*   Classify simple error patterns.

In [4]:
def diagnose(train_loss, valid_loss):
  if train_loss > 1.0 and valid_loss > 1.0:
    return "likely underfitting"
  if valid_loss > train_loss * 3:
    return "likely overfitting"
  return "roughly balanced"

diagnose(0.1, 0.8), diagnose(3.0, 3.5) # The first overfits (0.8 > 0.1 * 3), the second is underfitting

('likely overfitting', 'likely underfitting')

## 4. Step-by-step breakdown

*   The function is only a toy rule, not a universal test.
> *   High training and validation loss suggests the model cannot fit the pattern well.
> *   Very low training loss with much higher validation loss suggests overfitting.
> *   A balanced gap suggests neither obvious problem from these numbers alone.
*   Frontier labs generally monitor training and validation loss curves over time.
> * They pay attention to the validation loss relative to training loss;
> * especially when training loss continues decreasing while validation loss plateaus or starts increasing, which is a possible sign of overfitting (the model is learning training-specific patterns rather than generalizable patterns).

## 5. Connection to ML systems

*   Training dashboards often show training and validation curves.
*   The shape of these curves helps decide whether to train longer, simplify the model, add regularization, or get more data.

## 6. Common confusion points

- Overfitting is about the gap between training and new-data performance.
- Underfitting can come from too little training, not only too simple a model.
- A toy threshold is not a scientific diagnosis.
- Noisy validation estimates can mislead when validation data is small.

# 3.6.3 Model Selection

## 1. Intuition

*   Model selection means choosing between candidate models or settings.
*   A hyperparameter is a setting chosen before or outside training, such as learning rate, model size, or regularization strength.

## 2. Why this exists

*   Different choices can produce different generalization behavior.
*   We need a principled way to compare them without using test data repeatedly.

## 3. Examples

*   Choose the candidate with the lowest validation loss.

In [5]:
candidates = [
    {"degree": 1, "valid_loss": 0.8},
    {"degree": 2, "valid_loss": 0.3},
    {"degree": 5, "valid_loss": 0.6},
]

best = min(candidates, key=lambda item: item['valid_loss']) # key=lambda means "Use this function to decide how to compare items." For each candidate dictionary, extract its valid_loss value then calculate min()
best # degree 2

{'degree': 2, 'valid_loss': 0.3}

## 4. Step-by-step breakdown

*   Each dictionary stores one candidate setting and its validation loss.

*   `min(..., key=...)` chooses the item with the smallest selected value.

*   The `lambda` function tells Python to compare candidates by `valid_loss`.

*   The chosen model is selected by validation performance, not training performance.

## 5. Connection to ML systems

*   Model selection is common in experiments.
*   You try several learning rates, architectures, or regularization settings and choose using validation data.



## 6. Common confusion points

- A hyperparameter (e.g. lr, regularization) is not learned by ordinary gradient descent in the same run.
> * Hyperparameters are already defined before training begins, and they do not evolve during training.
- Validation data guides selection; test data estimates final performance.
- Reusing validation data many times can overfit the validation set.
- Lower validation loss is useful only if the validation set matches the real target problem.

# 3.6.4 Summary

## 1. Intuition

*   Generalization is the real goal of supervised learning.
*   Training error tells us how well the model fits known examples.
*   Validation or test error estimates performance on unseen examples.



## 2. Why this exists

*   A model is useful only if it handles future examples well enough for the task.

## 3. Examples

*   A compact checklist for generalization debugging.

In [ ]:
checks = [
    "compare train and validation loss",
    "watch for large gaps",
    "choose hyperparameters on validation data",
    "reserve test data for final estimate",
]

## 4. Step-by-step breakdown

*   The checklist follows the basic evaluation discipline.
> *   1) Compare train and validation behavior.
> *   2) Diagnose underfitting or overfitting.
> *   3) Choose settings carefully.
> *   4) Estimate performance on held-out data.

## 5. Connection to ML systems

*   This discipline becomes more important as models become more flexible and easier to overfit.
> * A more flexible model is easier to overfit because it has **more ability to fit both the true patterns and the noise in the training data**.
> * The larger model has more ways to represent relationships. That is useful because it can learn complex patterns, but it also means it can memorize irrelevant details.
*   There are a few ways to combat larger models overfitting:
> *   Diversity of training data (so its tougher to memorize all patterns)
> *   Weight decay (penalization on large weights, or intuitively, preferring simpler explanations)
>> * Without weight decay, the model may become "hyperactive" toward certain features and rely too heavily on them, ignoring more general patterns.
> *   Stop training once validation loss creep up despite training loss continuing to improve
> *   Pre-training + fine-tuning
> > *   Pre-training = General education (K-12 + broad foundational knowledge). The model learns language, concepts, patterns, and relationships from massive amounts of data.
> > *   Fine-tuning = Specialized education (college/professional training). The model adapts those existing capabilities toward specific tasks, behaviors, formats, or domains using higher-signal data.
> *   Dropout
>> * Randomly disable some neurons during training so the model learns redundant, robust features instead of relying on specific pathways

## 6. Common confusion points

- Generalization cannot be proven by training loss.
- More complex models can help or hurt.
> * They help because they have more capacity to learn complex patterns.
> * They can hurt because that same capacity allows them to fit noise and memorize training-specific details.
> * Concerns about scaling laws slowing down often refer to the difficulty of continuing to scale model size, compute, and high-quality data together.
> * If model capacity grows much faster than useful training data, the risk of inefficient training, memorization, and weaker generalization increases.
    ```
    Low capacity model:
        Cannot learn enough
        → underfitting

    High capacity model:
        Can learn everything
        → may learn useful patterns AND noise
    ```
- Evaluation protocol is part of the ML system.
- A single split gives only one estimate.

# 3.6.5 Exercises

## 1. Intuition

*   These exercises practice reading train/validation behavior.

## 2. Why this exists

*   Correct diagnosis prevents random trial-and-error changes.

## 3. Examples

*   Exercise 1: diagnose two loss patterns.

In [6]:
case_a = diagnose(train_loss=0.05, valid_loss=0.7)
case_b = diagnose(train_loss=2.0, valid_loss=2.5)
case_a, case_b #likely overfit, # likely underfit

('likely overfitting', 'likely underfitting')

*   Exercise 2: select the best learning rate by validation loss.

In [8]:
runs = [{'lr': 0.1, "valid_loss": 0.9}, {'lr': 0.01, "valid_loss":0.04}]
best_run = min(runs, key=lambda run: run["valid_loss"]) # This works exactly like key=lambda item: item["valid_loss"] because you already defined the collection argument 'runs' as the 1st argument of min()
best_run #0.04

{'lr': 0.01, 'valid_loss': 0.04}

## 4. Step-by-step breakdown

*   Exercise 1 checks underfitting and overfitting language.
*   Exercise 2 checks validation-based model selection.
*   The important habit is separating training performance from selection and final evaluation.



## 5. Connection to ML systems

*   This prepares for weight decay, where regularization changes the tradeoff between fitting and generalizing.

## 6. Common confusion points

- Validation loss is an estimate, not an absolute truth.
- The best validation run may not be best on future data.
- Do not choose based only on training loss.
- Record selection criteria before comparing many runs.